# Estimating average reservoir pressure in multi-fractured shale wells without extended shut-in
## Complete code: theory, simulator, synthetic data, estimators, physics-constrained learning and HFTS-1 application

This notebook reproduces every number, table and figure in the manuscript from the data in this package.
Run it from the `05_Code` folder (paths are relative to the package root).

| Section | What it does | Default run time |
| --- | --- | --- |
| 1 | Closed-form slab theory, Eqs. (5)–(14), and its numerical checks | seconds |
| 2 | Finite-volume element simulator, Eqs. (1)–(4), and its verification | under a minute |
| 3 | Synthetic dataset generation (switched off by default; the raw outputs are in `03_Raw_Data/synthetic`) | hours if switched on |
| 4 | Observation model, physics estimators and features: raw simulations → clean feature tables | minutes if switched on; a 20-case check always runs |
| 5 | Learning: model selection, five-fold case-grouped cross-validation, conformal intervals, out-of-distribution tests (Tables 2–3) | several minutes |
| 6 | HFTS-1: shut-in detection from raw gauge records, pad production from the RRC lease, Monte Carlo field estimates (Table 4) | a few minutes |
| 7 | All nine manuscript figures | about a minute |

Requirements: Python ≥ 3.10, numpy, scipy, pandas, scikit-learn 1.9 (the version used for the paper), matplotlib, openpyxl. See `requirements.txt`.

In [1]:
import os, sys, time, glob, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
PKG = Path("..").resolve()                     # package root
RAW, CLEAN = PKG / "03_Raw_Data", PKG / "04_Clean_Data"
OUT = PKG / "05_Code" / "outputs"
FIGS = OUT / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

# ---- switches
REGENERATE_SYNTHETIC = False   # True: rerun the simulator for all 3,356 cases (several hours)
REBUILD_FEATURE_TABLES = False # True: rebuild the four clean feature tables from the raw simulations (minutes)
RETRAIN = True                 # True: model selection + cross-validation + final models; False: load models_final.pkl
print("scikit-learn", sklearn.__version__, "| package root:", PKG)


def rcsv(path, **kw):
    """Read a packaged CSV with exact (round-trip) float parsing, so every number is bit-identical to the run."""
    return pd.read_csv(path, float_precision="round_trip", **kw)

scikit-learn 1.9.1 | package root: /home/claude/hfts1/package_build/HFTS1_Average_Pressure_Package


## 1. Closed-form buildup theory of the stimulated slab

Slab $0\le x\le L$ (fracture face at $x=0$, symmetry plane at $x=L$), $t_D=\eta t/L^2$, $\eta=k/(\phi\mu c_t)$.
Constant-$p_{wf}$ drawdown from $p_i$, then an ideal shut-in at $t_{pD}$:

* $p-p_{wf}=\Delta p_i\sum_m (2/a_m)\sin(a_m x_D)e^{-a_m^2 t_D}$, $a_m=(2m+1)\pi/2$  — Eq. (6)
* $B_n=4\Delta p_i\sum_m e^{-a_m^2 t_{pD}}/(a_m^2-n^2\pi^2)$ — Eq. (8)
* $R(\Delta t_D;t_{pD})=(p_{ws}-p_{wf})/(\bar p-p_{wf})$ — Eq. (9); universal limit $R_\infty$ — Eq. (10)

In [2]:
PI = np.pi


def a_m(M):
    return (2 * np.arange(M) + 1) * PI / 2


def pbar_minus_pwf(tpD, M=400):
    """Slab average minus p_wf at shut-in, per unit initial drawdown (Eq. 6)."""
    a = a_m(M)
    return np.sum(2.0 / a**2 * np.exp(-(a**2) * tpD))


def B_n(tpD, N=4000, M=400):
    """Buildup Fourier coefficients per unit initial drawdown (Eq. 8)."""
    a = a_m(M)
    n = np.arange(1, N + 1)
    w = np.exp(-(a**2) * tpD)
    return 4.0 * (w[None, :] / (a[None, :] ** 2 - (n[:, None] * PI) ** 2)).sum(axis=1)


def recovered_fraction(dtD, tpD, N=4000, M=400):
    """R(dt_D; t_pD), Eq. (9)."""
    dtD = np.atleast_1d(np.asarray(dtD, float))
    b = B_n(tpD, N, M)
    n = np.arange(1, N + 1)
    s = (b[None, :] * np.exp(-((n * PI) ** 2)[None, :] * dtD[:, None])).sum(axis=1)
    return 1.0 + s / pbar_minus_pwf(tpD, M)


def recovered_fraction_bdf(dtD, N=20000):
    """Universal law after boundary-dominated drawdown, Eq. (10)."""
    dtD = np.atleast_1d(np.asarray(dtD, float))
    n = np.arange(1, N + 1)
    c = 2.0 / (4 * n**2 - 1)
    return 1.0 - (c[None, :] * np.exp(-((n * PI) ** 2)[None, :] * dtD[:, None])).sum(axis=1)


# ---- checks quoted in Sections 3.1–3.2
from scipy.optimize import brentq
f_bdf = lambda x: float(recovered_fraction_bdf(np.array([x]))[0])
print("R(0) for t_pD = 0.05 (should be ~0):", float(recovered_fraction(1e-9, 0.05)[0]))
dt = np.geomspace(1e-6, 3, 3000)
for tp in (0.1, 0.2, 0.3, 0.5):
    print(f"t_pD = {tp}: max |R - R_inf| = {np.max(np.abs(recovered_fraction(dt, tp) - recovered_fraction_bdf(dt))):.5f}")
for r in (0.25, 0.5, 0.9, 0.99):
    print(f"R_inf = {r}: dt_D = {brentq(lambda x: f_bdf(x) - r, 1e-7, 3):.5f}")
x = np.geomspace(1e-6, 0.03, 200)
print("early asymptote (pi^1.5/2) sqrt(dt_D): max relative error up to dt_D = 0.03:",
      round(float(np.max(np.abs(PI**1.5 / 2 * np.sqrt(x) / recovered_fraction_bdf(x) - 1))), 4))
x = np.linspace(0.1, 2, 200)
print("late asymptote (2/3) exp(-pi^2 dt_D): max relative error from dt_D = 0.1:",
      round(float(np.max(np.abs(2 / 3 * np.exp(-PI**2 * x) / (1 - recovered_fraction_bdf(x)) - 1))), 4))
eta_hfts = 0.0002637 * 0.68e-3 / (0.07 * 0.4 * 1.5e-5)   # ft2/h, k in md, mu in cp
for L in (45, 90):
    d = eta_hfts * 72 / L**2
    print(f"HFTS-1 example, L = {L} ft: eta = {eta_hfts:.3f} ft2/h, dt_D(72 h) = {d:.4f}, R = {f_bdf(d):.3f}, "
          f"90 % recovery after {0.19229 * L**2 / eta_hfts / 24:.0f} days")

R(0) for t_pD = 0.05 (should be ~0): 0.00019724820934352572


t_pD = 0.1: max |R - R_inf| = 0.03662


t_pD = 0.2: max |R - R_inf| = 0.00510


t_pD = 0.3: max |R - R_inf| = 0.00071


t_pD = 0.5: max |R - R_inf| = 0.00001
R_inf = 0.25: dt_D = 0.00829
R_inf = 0.5: dt_D = 0.03632
R_inf = 0.9: dt_D = 0.19229
R_inf = 0.99: dt_D = 0.42552
early asymptote (pi^1.5/2) sqrt(dt_D): max relative error up to dt_D = 0.03: 0.0503
late asymptote (2/3) exp(-pi^2 dt_D): max relative error from dt_D = 0.1: 0.0103
HFTS-1 example, L = 45 ft: eta = 0.427 ft2/h, dt_D(72 h) = 0.0152, R = 0.335, 90 % recovery after 38 days
HFTS-1 example, L = 90 ft: eta = 0.427 ft2/h, dt_D(72 h) = 0.0038, R = 0.170, 90 % recovery after 152 days


### 1.1 Independent check: vertex-centred finite differences on the slab problem, Eq. (5)

In [3]:
def slab_fd(tpD, dts, NX=1600):
    x = np.linspace(0.0, 1.0, NX + 1)
    h = x[1] - x[0]
    vol = np.full(NX + 1, h); vol[0] = vol[-1] = h / 2
    main = np.zeros(NX + 1); off = np.full(NX, -1.0 / h)
    main[:-1] += 1.0 / h; main[1:] += 1.0 / h
    K = sp.diags([off, main, off], [-1, 0, 1], format="csc")

    def step(p, dt, dirichlet0):
        A = (sp.diags(vol / dt) + K).tolil()
        b = vol / dt * p
        if dirichlet0 is not None:
            A[0, :] = 0; A[0, 0] = 1.0; b[0] = dirichlet0
        return spla.spsolve(A.tocsc(), b)

    p = np.ones(NX + 1)
    for dt in np.diff(np.concatenate([[0], np.geomspace(1e-7, tpD, 400)])):
        p = step(p, dt, 0.0)
    pbar = np.sum(vol * p)
    grid = np.concatenate([[0], np.geomspace(1e-7, dts.max(), 600)])
    ws = []
    for i in range(1, len(grid)):
        p = step(p, grid[i] - grid[i - 1], None)
        ws.append(p[0])
    return pbar, np.interp(dts, grid[1:], np.array(ws))


dts = np.array([1e-4, 1e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0])
worst = 0.0
for tpD in (0.01, 0.05, 0.2, 1.0):
    pbar, ws = slab_fd(tpD, dts)
    worst = max(worst, np.max(np.abs(ws / pbar - recovered_fraction(dts, tpD))))
print("max |R_FD - R_series| over four production times:", round(worst, 4))

max |R_FD - R_series| over four production times: 0.002


## 2. Finite-volume simulator of the closed MFHW element

Plan-view element $x\in[0,L]$, $y\in[0,Y_e]$; fracture half-width column for $y<x_f$; no-flow outer boundaries.
Field units; transmissibility $0.001127\,kA/(\mu\Delta)$ bbl/(d·psi); accumulation $V\phi c_t/5.615$ bbl/psi;
fully implicit; optional $c_t(p)$ via Picard iterations on the mass function; wellbore node with storage $C_w$.

In [4]:
C1, C2 = 0.001127, 5.615


def graded(x0, x1, h0, r, hmax=None):
    """Cell faces from x0 to x1 with first cell h0 growing by ratio r."""
    f = [x0]
    h = h0
    while f[-1] + h < x1 - 1e-12:
        f.append(f[-1] + h)
        h = h * r if hmax is None else min(h * r, hmax)
    if x1 - f[-1] < 0.3 * (f[-1] - f[-2]) and len(f) > 2:
        f[-1] = x1
    else:
        f.append(x1)
    return np.array(f)


class Element:
    def __init__(self, L, xf, Ye, h, k, phi, mu, ct, w=0.02, kf=None, Fc=10.0, phif=0.3, Cw=0.0,
                 ct_fun=None, nx_r=1.25, ny_r=1.3):
        self.L, self.xf, self.Ye, self.h = L, xf, Ye, h
        self.k, self.phi, self.mu, self.ct = k, phi, mu, ct
        self.ct_fun = ct_fun
        hw = w / 2.0
        kf = Fc / w if kf is None else kf
        xm = graded(hw, L, 0.01, nx_r)
        self.xfc = np.concatenate([[0.0], xm])
        if xf < Ye - 1e-9:
            y1 = graded(0.0, xf / 2, 0.5, ny_r)
            y2 = xf - graded(0.0, xf / 2, 0.5, ny_r)[::-1]
            y3 = graded(xf, Ye, 0.5, ny_r)
            self.yfc = np.unique(np.concatenate([y1, y2, y3]).round(9))
        else:
            self.yfc = graded(0.0, Ye, 0.5, ny_r)
        nx, ny = len(self.xfc) - 1, len(self.yfc) - 1
        self.nx, self.ny = nx, ny
        dx, dy = np.diff(self.xfc), np.diff(self.yfc)
        xc = 0.5 * (self.xfc[:-1] + self.xfc[1:]); yc = 0.5 * (self.yfc[:-1] + self.yfc[1:])
        self.xc, self.yc, self.dx, self.dy = xc, yc, dx, dy
        X, Y = np.meshgrid(xc, yc, indexing="ij")
        self.is_frac = np.zeros((nx, ny), bool)
        self.is_frac[0, :] = yc < xf
        kk = np.full((nx, ny), k, float); kk[self.is_frac] = kf
        ph = np.full((nx, ny), phi, float); ph[self.is_frac] = phif
        self.kk, self.ph = kk, ph
        V = np.outer(dx, dy) * h
        self.V = V
        self.pv = V * ph
        self.srv = (Y < xf)  # stimulated slab
        n = nx * ny
        idx = np.arange(n).reshape(nx, ny)
        rows, cols, vals = [], [], []
        Tx = C1 * dy[None, :] * h / mu / (0.5 * dx[:-1, None] / kk[:-1, :] + 0.5 * dx[1:, None] / kk[1:, :])
        Ty = C1 * dx[:, None] * h / mu / (0.5 * dy[None, :-1] / kk[:, :-1] + 0.5 * dy[None, 1:] / kk[:, 1:])
        for (a, b, T) in [(idx[:-1, :], idx[1:, :], Tx), (idx[:, :-1], idx[:, 1:], Ty)]:
            a, b, T = a.ravel(), b.ravel(), T.ravel()
            rows += [a, b, a, b]; cols += [a, b, b, a]; vals += [T, T, -T, -T]
        Tf0 = C1 * kk[0, 0] * dy[0] * h / mu / (0.5 * dx[0])
        self.Tw = 50.0 * Tf0
        c0 = idx[0, 0]
        rows += [np.array([c0, n, c0, n])]; cols += [np.array([c0, n, n, c0])]
        vals += [np.array([self.Tw, self.Tw, -self.Tw, -self.Tw])]
        self.K = sp.csc_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(n + 1, n + 1))
        self.n, self.Cw, self.c0 = n, Cw, c0
        self.acc_v = (self.pv / C2).ravel()

    def ct_of(self, p):
        return np.full(p.shape, self.ct) if self.ct_fun is None else self.ct_fun(p)

    def m_of(self, p, pref):
        """Mass function m(p) = int_pref^p c_t dp."""
        if self.ct_fun is None:
            return self.ct * (p - pref)
        grid = np.linspace(min(p.min(), pref) - 1, max(p.max(), pref) + 1, 4001)
        c = self.ct_fun(grid)
        M = np.concatenate([[0], np.cumsum(0.5 * (c[1:] + c[:-1]) * np.diff(grid))])
        return np.interp(p, grid, M) - np.interp(pref, grid, M)

    def averages(self, p, pref):
        """(slab average, element average); mass-equivalent (stabilised) pressures, Eqs. (3)-(4)."""
        pc = p[: self.n].reshape(self.nx, self.ny)
        out = []
        for mask in (self.srv, np.ones_like(self.srv)):
            w = self.pv[mask]
            if self.ct_fun is None:
                out.append(float(np.sum(w * pc[mask]) / np.sum(w)))
            else:
                mbar = np.sum(w * self.m_of(pc[mask], pref)) / np.sum(w)
                lo, hi = pc[mask].min() - 1, pc[mask].max() + 1
                for _ in range(60):
                    mid = 0.5 * (lo + hi)
                    if self.m_of(np.array([mid]), pref)[0] < mbar:
                        lo = mid
                    else:
                        hi = mid
                out.append(0.5 * (lo + hi))
        return out

    def _solve(self, p_old, dt, pw_bc, cache):
        n = self.n
        if self.ct_fun is None:
            key = (round(dt, 12), pw_bc is None)
            if key not in cache:
                acc = np.concatenate([self.acc_v * self.ct, [self.Cw]]) / dt
                A = (self.K + sp.diags(acc)).tolil()
                if pw_bc is not None:
                    A[n, :] = 0; A[n, n] = 1.0
                if pw_bc is None and self.Cw == 0.0:
                    A[n, n] += 1e-12
                cache[key] = (spla.splu(A.tocsc()), acc)
            lu, acc = cache[key]
            b = acc * p_old
            if pw_bc is not None:
                b[n] = pw_bc
            return lu.solve(b)
        p = p_old.copy()
        for _ in range(8):
            c = np.concatenate([self.acc_v * self.ct_of(p[:n]), [self.Cw]]) / dt
            mres = np.concatenate([self.acc_v * (self.m_of(p[:n], 0) - self.m_of(p_old[:n], 0)), [self.Cw * (p[n] - p_old[n])]]) / dt
            A = (self.K + sp.diags(c)).tolil()
            b = c * p - mres
            if pw_bc is not None:
                A[n, :] = 0; A[n, n] = 1.0; b[n] = pw_bc
            p_new = spla.spsolve(A.tocsc(), b)
            if np.max(np.abs(p_new - p)) < 1e-4:
                p = p_new
                break
            p = p_new
        return p

    def run(self, pi, pwf_fun, tp_days, shut_hours, dt_max=2.0):
        """Produce under p_wf(t) for tp_days, then shut in; report p_ws at shut_hours (h)."""
        n = self.n
        p = np.full(n + 1, float(pi))
        cache = {}
        t, dt = 0.0, 1e-4
        T, PW, Q, QS, PBS, PBE = [0.0], [pi], [0.0], [0.0], [pi], [pi]
        while t < tp_days - 1e-12:
            dt = min(dt, dt_max, tp_days - t)
            pw = pwf_fun(t + dt)
            pw_old = p[n]
            p = self._solve(p, dt, pw, cache)
            t += dt
            qsf = self.Tw * (p[self.c0] - p[n])
            qs = qsf - self.Cw * (p[n] - pw_old) / dt
            a = self.averages(p, pi) if len(T) % 5 == 0 or t >= tp_days - 1e-12 else (np.nan, np.nan)
            T.append(t); PW.append(p[n]); Q.append(qsf); QS.append(qs); PBS.append(a[0]); PBE.append(a[1])
            dt *= 1.2
        pbar_srv, pbar_el = self.averages(p, pi)
        p_shut = p.copy()
        sh = np.asarray(shut_hours, float) / 24.0
        grid = np.unique(np.concatenate([np.geomspace(1e-5, sh.max(), 160), sh]))
        tt, pws = 0.0, []
        cache2 = {}
        for tg in grid:
            dt = tg - tt
            if dt <= 0:
                continue
            p = self._solve(p, dt, None, cache2 if self.ct_fun is None else None)
            tt = tg
            pws.append(p[n])
        pws = np.interp(sh, grid[: len(pws)], np.array(pws))
        return dict(t=np.array(T), pwf=np.array(PW), q=np.array(Q), qs=np.array(QS),
                    pbar_srv_t=np.array(PBS), pbar_el_t=np.array(PBE), pbar_srv=pbar_srv, pbar_el=pbar_el,
                    shut_hours=np.asarray(shut_hours, float), pws=pws, p_end_state=p, p_shut_state=p_shut)

### 2.1 Verification (Section 4.2): slab limit, mass balance and grid convergence

In [5]:
k_, phi_, mu_, ct_ = 6.8e-4, 0.07, 0.4, 1.5e-5
eta_d = 0.006328 * k_ / (phi_ * mu_ * ct_)    # ft2/day
sh = np.geomspace(0.05, 3000, 40)
for Lv, tp in ((30.0, 60.0), (45.0, 150.0), (60.0, 400.0)):
    el = Element(L=Lv, xf=200.0, Ye=200.0, h=100.0, k=k_, phi=phi_, mu=mu_, ct=ct_, Fc=1e5, Cw=0.0, nx_r=1.12, ny_r=1.3)
    r = el.run(5000.0, lambda t: 1000.0, tp, sh, dt_max=0.1)
    Le = Lv - 0.01
    tpD = eta_d * tp / Le**2
    Rn = (r["pws"] - 1000.0) / (r["pbar_srv"] - 1000.0)
    th = 1000 + 4000 * pbar_minus_pwf(tpD)
    print(f"slab limit L={Lv:.0f} ft, t_pD={tpD:.2f}: max|R_num - R_series| = {np.max(np.abs(Rn - recovered_fraction(eta_d * sh / 24 / Le**2, tpD))):.4f}; "
          f"pbar num {r['pbar_srv']:.1f} vs series {th:.1f} psia")
# mass balance with finite conductivity, storage, outer region and declining BHP
el = Element(L=45.0, xf=150.0, Ye=330.0, h=100.0, k=k_, phi=phi_, mu=mu_, ct=ct_, Fc=5.0, Cw=0.05)
pwf_f = lambda t: 900 + 3500 * np.exp(-t / 60.0)
r = el.run(5000.0, pwf_f, 220.0, np.array([1.0, 24.0, 72.0]))
Qs = np.sum(r["qs"][1:] * np.diff(r["t"]))
dep = np.sum(el.acc_v * ct_ * (5000.0 - r["p_shut_state"][: el.n])) + el.Cw * (5000.0 - r["p_shut_state"][el.n])
print(f"mass balance: surface cumulative {Qs:.4f} bbl vs depletion {dep:.4f} bbl, relative difference {abs(Qs - dep) / dep:.2e}")
for nxr, nyr in ((1.4, 1.5), (1.25, 1.3), (1.12, 1.15)):
    el = Element(L=45.0, xf=150.0, Ye=330.0, h=100.0, k=k_, phi=phi_, mu=mu_, ct=ct_, Fc=5.0, Cw=0.05, nx_r=nxr, ny_r=nyr)
    r = el.run(5000.0, pwf_f, 220.0, np.array([0.1, 1, 10, 24, 72, 168]))
    print(f"grid {el.n:5d} cells: pbar_S = {r['pbar_srv']:.2f} psia, p_ws = " + " ".join(f"{v:.2f}" for v in r["pws"]))

slab limit L=30 ft, t_pD=0.68: max|R_num - R_series| = 0.0099; pbar num 1603.2 vs series 1600.4 psia


slab limit L=45 ft, t_pD=0.76: max|R_num - R_series| = 0.0094; pbar num 1499.9 vs series 1498.0 psia


slab limit L=60 ft, t_pD=1.14: max|R_num - R_series| = 0.0101; pbar num 1196.6 vs series 1195.2 psia


mass balance: surface cumulative 694.5096 bbl vs depletion 694.5096 bbl, relative difference 1.10e-08


grid   840 cells: pbar_S = 1732.83 psia, p_ws = 989.57 990.48 999.26 1012.07 1051.38 1117.42


grid  1584 cells: pbar_S = 1727.43 psia, p_ws = 989.57 990.48 999.26 1012.07 1051.36 1117.33


grid  4275 cells: pbar_S = 1724.17 psia, p_ws = 989.57 990.48 999.26 1012.07 1051.35 1117.27


## 3. Synthetic dataset (Section 4.3, Table 1)

`sample()` draws one case; `run_case()` simulates it and returns well-level outputs (element rate × number of elements).
The 3,356 simulated cases are packaged in `03_Raw_Data/synthetic/{main,nonlinear,extrap}`; set `REGENERATE_SYNTHETIC = True` to rerun them.
Seeds: main set cases 0–2999 with seed 20261024 (cases 2541, 2931 and 2997 failed grid construction), nonlinear cases 0–159 with seed 777, extrapolation cases 0–199 with seed 999 (case 128 failed).

In [6]:
SHUT_H = np.unique(np.concatenate([np.geomspace(0.01, 504.0, 90), [24.0, 48.0, 72.0, 168.0]]))


def loguni(rng, a, b):
    return float(np.exp(rng.uniform(np.log(a), np.log(b))))


def sample(rng, mode):
    p = {}
    p["k"] = loguni(rng, 1e-4, 3e-3)
    p["phi"] = rng.uniform(0.04, 0.10)
    p["mu"] = loguni(rng, 0.2, 1.0)
    p["ct"] = loguni(rng, 5e-6, 3e-5)
    p["L"] = loguni(rng, 15.0, 150.0)
    p["xf"] = rng.uniform(75.0, 400.0)
    p["Ye"] = float(np.clip(p["xf"] * rng.uniform(1.0, 2.5), 100.0, 600.0))
    p["Ye"] = max(p["Ye"], p["xf"])
    p["h"] = rng.uniform(50.0, 150.0)
    p["Fc"] = loguni(rng, 0.5, 50.0)
    p["lateral"] = rng.uniform(7500.0, 10000.0)
    p["n_elem"] = 4.0 * p["lateral"] / (2.0 * p["L"])
    p["C_well"] = loguni(rng, 0.01, 1.0)
    p["Cw"] = p["C_well"] / p["n_elem"]
    p["pi"] = rng.uniform(4000.0, 6500.0)
    p["tp"] = loguni(rng, 60.0, 900.0)
    p["p_start"] = p["pi"] - rng.uniform(200.0, 1500.0)
    p["p_end"] = rng.uniform(300.0, 1500.0)
    p["tau"] = p["tp"] * rng.uniform(0.1, 0.6)
    nst = rng.integers(0, 5)
    p["steps_t"] = list(rng.uniform(0.05, 0.9, nst) * p["tp"])
    p["steps_a"] = list(rng.choice([-1, 1], nst) * rng.uniform(30.0, 150.0, nst))
    if mode == "extrap":
        p["L"] = loguni(rng, 150.0, 300.0)
        p["k"] = loguni(rng, 3e-3, 1e-2)
        p["n_elem"] = 4.0 * p["lateral"] / (2.0 * p["L"])
        p["Cw"] = p["C_well"] / p["n_elem"]
    if mode == "nonlinear":
        p["pb"] = rng.uniform(1500.0, 3500.0)
        p["ct_gas_mult"] = rng.uniform(3.0, 10.0)
    return p


def pwf_fun_factory(p):
    st, sa = np.array(p["steps_t"]), np.array(p["steps_a"])

    def f(t):
        base = p["p_end"] + (p["p_start"] - p["p_end"]) * np.exp(-t / p["tau"])
        s = np.sum(sa * 0.5 * (1 + np.tanh((t - st) / 0.5))) if len(st) else 0.0
        return float(np.clip(base + s, 150.0, p["pi"] - 50.0))
    return f


def ct_fun_factory(p):
    """Eq. (15): compressibility rising below a bubble point (nonlinear set)."""
    if "pb" not in p:
        return None
    ct0, pb, m = p["ct"], p["pb"], p["ct_gas_mult"]
    return lambda pr: ct0 * (1.0 + (m - 1.0) * 0.5 * (1 + np.tanh((pb - pr) / 150.0)) * np.clip((pb - pr) / pb + 0.3, 0.3, 1.3))


def run_case(i, seed, mode):
    rng = np.random.default_rng(seed + 7919 * i)
    p = sample(rng, mode)
    el = Element(L=p["L"], xf=p["xf"], Ye=p["Ye"], h=p["h"], k=p["k"], phi=p["phi"], mu=p["mu"], ct=p["ct"],
                 Fc=p["Fc"], Cw=p["Cw"], ct_fun=ct_fun_factory(p))
    r = el.run(p["pi"], pwf_fun_factory(p), p["tp"], SHUT_H, dt_max=2.0)
    s = p["n_elem"]
    return dict(params=p, t=r["t"], pwf=r["pwf"], q_well=r["q"] * s, qs_well=r["qs"] * s,
                pbar_srv_t=r["pbar_srv_t"], pbar_el_t=r["pbar_el_t"], pbar_srv=r["pbar_srv"], pbar_el=r["pbar_el"],
                shut_h=SHUT_H, pws=r["pws"], cells=el.n, mode=mode)


def load_raw_cases(mode):
    """Rebuild the simulator output dictionaries from the packaged raw CSV files."""
    d = RAW / "synthetic" / mode
    P = rcsv(d / "cases_parameters_and_targets.csv")
    PH = rcsv(d / "production_histories.csv")
    BU = rcsv(d / "buildups.csv")
    gp, gb = dict(tuple(PH.groupby("case"))), dict(tuple(BU.groupby("case")))
    cases = {}
    for _, r in P.iterrows():
        c = int(r.case)
        h, b = gp[c], gb[c]
        params = {k: r[k] for k in P.columns if k not in ("set", "case", "pbar_S_psia", "pbar_E_psia", "cells",
                                                          "choke_steps_t_days", "choke_steps_dp_psi") and pd.notna(r[k])}
        cases[c] = dict(params=params, t=h.t_days.values, pwf=h.pwf_psia.values, q_well=h.q_sandface_bbl_d.values,
                        qs_well=h.q_surface_bbl_d.values, pbar_srv=r.pbar_S_psia, pbar_el=r.pbar_E_psia,
                        shut_h=b.dt_h.values, pws=b.pws_psia.values, cells=r.cells, mode=mode)
    return cases


# reproduce one packaged case from its seed (demonstrates that the raw data are regenerable)
t0 = time.time()
demo = run_case(0, 20261024, "main")
raw0 = load_raw_cases("main")[0]
print(f"case 0 regenerated in {time.time() - t0:.1f} s: pbar_S {demo['pbar_srv']:.4f} vs packaged {raw0['pbar_srv']:.4f} psia; "
      f"max |p_ws difference| = {np.max(np.abs(demo['pws'] - raw0['pws'])):.2e} psi")
if REGENERATE_SYNTHETIC:
    for mode, seed, count in (("main", 20261024, 3000), ("nonlinear", 777, 160), ("extrap", 999, 200)):
        out = OUT / "regenerated" / mode
        out.mkdir(parents=True, exist_ok=True)
        for i in range(count):
            try:
                pickle.dump(run_case(i, seed, mode), open(out / f"case_{i:05d}.pkl", "wb"))
            except Exception as e:
                print(mode, i, "failed:", repr(e))

case 0 regenerated in 12.0 s: pbar_S 3481.1769 vs packaged 3481.1769 psia; max |p_ws difference| = 0.00e+00 psi


## 4. Observation model, physics estimators and features (Sections 4.4 and 5.1–5.2)

In [7]:
_LOGTD = np.linspace(-7, 1.5, 600)
_RBDF = recovered_fraction_bdf(10.0**_LOGTD, N=6000)


def R_bdf(dtD):
    return np.interp(np.log10(np.clip(np.asarray(dtD, float), 1e-7, 10**1.5)), _LOGTD, _RBDF)


def observe(case, N, rng, p_i_err=0.08, rate_err=0.05, gauge_sd=(0.1, 1.0), pwf_sd=1.0):
    """Field-like observations of a simulated case: noisy monthly rates, BHP history, p_i, N-hour buildup."""
    t, pwf, q = case["t"], case["pwf"], case["q_well"]
    tp = t[-1]
    Qc = np.concatenate([[0.0], np.cumsum(q[1:] * np.diff(t))])
    edges = np.arange(0.0, tp + 1e-9, 30.0)
    if edges[-1] < tp - 1e-9:
        edges = np.append(edges, tp)
    Qe = np.interp(edges, t, Qc)
    dur = np.diff(edges)
    keep = dur >= 5.0
    qm = (np.diff(Qe) / dur)[keep] * (1 + rate_err * rng.standard_normal(keep.sum()))
    tm_end = edges[1:][keep]
    tm_mid = 0.5 * (edges[:-1] + edges[1:])[keep]
    pwm = np.array([np.mean(np.interp(np.linspace(a, b, 20), t, pwf)) for a, b in zip(edges[:-1][keep], edges[1:][keep])])
    Qm = np.cumsum(qm * dur[keep])
    tdays = np.linspace(0, tp, 200)
    pwd = np.interp(tdays, t, pwf) + pwf_sd * rng.standard_normal(tdays.size)
    sh, pws = case["shut_h"], case["pws"]
    m = sh <= N + 1e-9
    sd = rng.uniform(*gauge_sd)
    bu_t = sh[m]
    bu_p = np.round(pws[m] + sd * rng.standard_normal(m.sum()), 1)
    return dict(tp=tp, pwf_end=float(pwf[-1]), pi_obs=float(case["params"]["pi"] * (1 + p_i_err * rng.standard_normal())),
                qm=qm, Qm=Qm, tm_end=tm_end, tm_mid=tm_mid, pwm=pwm, tdays=tdays, pwd=pwd,
                bu_t=bu_t, bu_p=bu_p, N=float(N), Q_total=float(Qc[-1] * (1 + rate_err * 0.3 * rng.standard_normal())))


def est_proxy(o):
    return float(np.interp(o["N"], o["bu_t"], o["bu_p"]))


def est_horner(o):
    tpe = 24.0 * o["Q_total"] / max(o["qm"][-1], 1e-9)
    w = (o["bu_t"] >= o["N"] / 10) & (o["bu_t"] <= o["N"])
    if w.sum() < 4:
        return np.nan
    b, a = np.polyfit(np.log10((tpe + o["bu_t"][w]) / o["bu_t"][w]), o["bu_p"][w], 1)
    return float(a)


def fmb(o, frac=0.5):
    """Flowing material balance, Eq. (13): returns (pbar, D [1/day], slope, intercept)."""
    q, Q, pw = o["qm"], o["Qm"], o["pwm"]
    ok = q > 0
    q, Q, pw = q[ok], Q[ok], pw[ok]
    n = len(q)
    if n < 3:
        return np.nan, np.nan, np.nan, np.nan
    k0 = max(0, min(int(n * (1 - frac)), n - 3))
    rnp = (o["pi_obs"] - pw[k0:]) / q[k0:]
    te = Q[k0:] / q[k0:]
    m, b = np.polyfit(te, rnp, 1)
    if not (m > 0 and b > 0):
        return np.nan, np.nan, m, b
    return float(o["pi_obs"] - o["Q_total"] * m), float(m / b), m, b


def _window(o):
    return (o["bu_t"] >= max(2.0, o["N"] / 10)) & (o["bu_t"] <= o["N"])


def est_sqrt(o, D):
    """Square-root-decline estimator, Eq. (14)."""
    if not (D > 0):
        return np.nan
    w = _window(o)
    if w.sum() < 4:
        return np.nan
    msq, c = np.polyfit(np.sqrt(o["bu_t"][w] / 24.0), o["bu_p"][w], 1)
    return float(o["pwf_end"] + msq / np.sqrt(np.pi * D)) if msq > 0 else np.nan


def est_slab(o, D):
    """Slab inversion: amplitude fit of Eq. (10) with tau_L = pi^2/(4D)."""
    if not (D > 0):
        return np.nan
    tau = np.pi**2 / (4 * D) * 24.0
    w = _window(o)
    if w.sum() < 4:
        return np.nan
    r = R_bdf(o["bu_t"][w] / tau)
    A = float(np.dot(r, o["bu_p"][w] - o["pwf_end"]) / np.dot(r, r))
    return o["pwf_end"] + A if A > 0 else np.nan


SHAPE_FR = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7])


def _z(p, pwf, pN):
    d = (p - pwf) / (pN - pwf)
    return float(np.log(d)) if (np.isfinite(d) and d > 0) else np.nan


def features(o):
    pN = est_proxy(o)
    pwf = o["pwf_end"]
    dN = pN - pwf
    shape = np.interp(np.log(o["N"] * SHAPE_FR), np.log(o["bu_t"]), (o["bu_p"] - pwf) / dN)
    w = o["bu_t"] >= o["N"] / 3
    dp = np.clip(o["bu_p"][w] - pwf, 1e-3, None)
    expo = np.polyfit(np.log(o["bu_t"][w]), np.log(dp), 1)[0] if w.sum() >= 3 else np.nan
    qn = o["qm"] / np.max(o["qm"])
    last = np.full(6, np.nan)
    k = min(6, len(qn))
    last[6 - k:] = qn[-k:]
    pwh = (np.interp(o["tp"] * np.array([0.25, 0.5, 0.75]), o["tdays"], o["pwd"]) - pwf) / max(o["pi_obs"] - pwf, 1.0)
    p_fmb, D, m, b = fmb(o)
    p_h, p_sq, p_sl = est_horner(o), est_sqrt(o, D), est_slab(o, D)
    data = dict(logN=np.log(o["N"]), logtp=np.log(o["tp"]), expo=expo, log_pi_ratio=np.log(max(o["pi_obs"] - pwf, 1.0) / dN),
                n_months=len(qn), **{f"s{j}": v for j, v in enumerate(shape)}, **{f"q{j}": v for j, v in enumerate(last)},
                **{f"pw{j}": v for j, v in enumerate(pwh)})
    phys = dict(logDN=np.log(D * o["N"] / 24) if D > 0 else np.nan, logDtp=np.log(D * o["tp"]) if D > 0 else np.nan,
                z_horner=_z(p_h, pwf, pN), z_fmb=_z(p_fmb, pwf, pN), z_sqrt=_z(p_sq, pwf, pN), z_slab=_z(p_sl, pwf, pN))
    return data, phys, dict(proxy=pN, horner=p_h, fmb=p_fmb, sqrt=p_sq, slab=p_sl, D=D), pwf, pN


def build_rows(cases, seed, pi_err=0.08):
    """One row per (case, N); seeds used for the paper: main 1, nonlinear 5, extrapolation 3, main with 15 % p_i error 1."""
    rows = []
    for cid in sorted(cases):
        c = cases[cid]
        p = c["params"]
        for N in (24, 72, 168):
            rng = np.random.default_rng(seed * 100003 + cid * 7 + N)
            o = observe(c, N, rng, p_i_err=pi_err)
            d, ph, e, pwf, pN = features(o)
            if not (pN - pwf > 1.0):
                continue
            eta = 0.006328 * p["k"] / (p["phi"] * p["mu"] * p["ct"])
            rows.append(dict(case=cid, N=N, pwf=pwf, pN=pN, pbar_srv=c["pbar_srv"], pbar_el=c["pbar_el"], pi=p["pi"],
                             z=float(np.log((c["pbar_srv"] - pwf) / (pN - pwf))) if c["pbar_srv"] > pwf else np.nan,
                             tauL_h=24 * p["L"] ** 2 / eta, dtD=N / 24 * eta / p["L"] ** 2, tpD=eta * o["tp"] / p["L"] ** 2,
                             **{f"par_{k}": v for k, v in p.items() if np.isscalar(v)}, **d, **ph,
                             **{f"est_{k}": v for k, v in e.items()}))
    return pd.DataFrame(rows)


TABLES = {n: rcsv(CLEAN / "synthetic" / f"feature_table_{n}.csv") for n in ("main", "nonlinear", "extrap", "main_pierr15")}
for n, T in TABLES.items():
    print(f"clean table {n:13s}: {len(T):5d} rows, {T.case.nunique():5d} cases")
# check: rebuild 20 main cases from the raw simulations and compare with the packaged clean table
raw_main = load_raw_cases("main")
sub = {c: raw_main[c] for c in sorted(raw_main)[:20]}
Rb = build_rows(sub, seed=1)
Tc = TABLES["main"].merge(Rb[["case", "N", "z", "est_sqrt", "est_horner"]], on=["case", "N"], suffixes=("", "_rebuilt"))
print("rebuilt vs packaged, max |difference|: z", float(np.nanmax(np.abs(Tc.z - Tc.z_rebuilt))),
      "| sqrt estimate", float(np.nanmax(np.abs(Tc.est_sqrt - Tc.est_sqrt_rebuilt))), "psi")
if REBUILD_FEATURE_TABLES:
    TABLES["main"] = build_rows(raw_main, seed=1)
    TABLES["main_pierr15"] = build_rows(raw_main, seed=1, pi_err=0.15)
    TABLES["nonlinear"] = build_rows(load_raw_cases("nonlinear"), seed=5)
    TABLES["extrap"] = build_rows(load_raw_cases("extrap"), seed=3)

clean table main         :  8942 rows,  2985 cases
clean table nonlinear    :   480 rows,   160 cases
clean table extrap       :   597 rows,   199 cases
clean table main_pierr15 :  8942 rows,  2985 cases


rebuilt vs packaged, max |difference|: z 0.0 | sqrt estimate 0.0 psi


### 4.1 How wrong is a short shut-in? (Section 6.1)

In [8]:
T = TABLES["main"]
for N in (24, 72, 168):
    s = T[T.N == N]
    R = (s.pN - s.pwf) / (s.pbar_srv - s.pwf)
    print(f"N = {N:3d} h: median dt_D {s.dtD.median():.3f}, median R {R.median():.3f}, median underestimate "
          f"{(s.pbar_srv - s.pN).median():.0f} psi, within 10 % {(abs(R - 1) <= 0.1).mean():.3f}, R > 1 in {(R > 1).mean():.3f}")
d1 = T.drop_duplicates("case")
print(f"median deficit {(d1.pbar_srv - d1.pwf).median():.0f} psi; pbar_E - pbar_S median {(d1.pbar_el - d1.pbar_srv).median():.0f} psi "
      f"(IQR {(d1.pbar_el - d1.pbar_srv).quantile(0.25):.0f}-{(d1.pbar_el - d1.pbar_srv).quantile(0.75):.0f})")
eta_c = 0.0002637 * d1.par_k / (d1.par_phi * d1.par_mu * d1.par_ct)
ratio = d1.est_D / (np.pi**2 * eta_c / (4 * d1.par_L**2) * 24)
for lo, hi in ((0, 0.1), (0.1, 0.3), (0.3, 1), (1, 3), (3, 1e9)):
    m = (d1.tpD >= lo) & (d1.tpD < hi)
    print(f"t_pD in [{lo}, {hi}): fitted/true decline constant, median {np.nanmedian(ratio[m]):.2f} "
          f"(IQR {np.nanpercentile(ratio[m], 25):.2f}-{np.nanpercentile(ratio[m], 75):.2f}), n = {m.sum()}")

N =  24 h: median dt_D 0.004, median R 0.288, median underestimate 858 psi, within 10 % 0.012, R > 1 in 0.012
N =  72 h: median dt_D 0.012, median R 0.421, median underestimate 695 psi, within 10 % 0.049, R > 1 in 0.058
N = 168 h: median dt_D 0.028, median R 0.551, median underestimate 531 psi, within 10 % 0.070, R > 1 in 0.127
median deficit 1214 psi; pbar_E - pbar_S median 537 psi (IQR 220-990)
t_pD in [0, 0.1): fitted/true decline constant, median 16.03 (IQR 8.67-39.68), n = 398
t_pD in [0.1, 0.3): fitted/true decline constant, median 2.93 (IQR 2.11-4.76), n = 479
t_pD in [0.3, 1): fitted/true decline constant, median 1.28 (IQR 0.95-1.84), n = 642
t_pD in [1, 3): fitted/true decline constant, median 0.80 (IQR 0.56-1.10), n = 630
t_pD in [3, 1000000000.0): fitted/true decline constant, median 0.27 (IQR 0.13-0.56), n = 836


## 5. Physics-constrained learning and validation (Sections 5.2–5.3, Tables 2–3)

Target $z=\ln[(\bar p_S-p_{wf})/(p_{ws}(N)-p_{wf})]$ (Eq. 16). ML-D: data features only. ML-P: residual on the physics base (Eq. 17).
Conformalised quantile regression gives 80 % intervals (Eq. 18). All splits are by case.

In [9]:
DATA = ["logN", "logtp", "expo", "log_pi_ratio", "n_months"] + [f"s{j}" for j in range(8)] + [f"q{j}" for j in range(6)] + [f"pw{j}" for j in range(3)]
PHYS = ["logDN", "logDtp", "z_horner", "z_fmb", "z_sqrt", "z_slab"]
PHYS_EST = ["horner", "fmb", "sqrt", "slab"]
EST = ["proxy"] + PHYS_EST


def rel_err(p, d):
    return (p - d.pbar_srv) / (d.pbar_srv - d.pwf)


def fit(X, y, params, loss="squared_error", q=None):
    kw = dict(max_iter=1500, learning_rate=params["lr"], max_leaf_nodes=params["leaves"], min_samples_leaf=params["msl"],
              l2_regularization=params["l2"], early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=0)
    if loss == "quantile":
        kw.update(loss="quantile", quantile=q)
    return HGB(**kw).fit(X, y)


df = TABLES["main"]
df = df[np.isfinite(df.z)].reset_index(drop=True)
if RETRAIN:
    # ---- model selection on one case split (70/15/15)
    rng = np.random.default_rng(42)
    cases = np.array(sorted(df.case.unique()))
    rng.shuffle(cases)
    n = len(cases)
    tr_c, va_c = set(cases[: int(0.7 * n)]), set(cases[int(0.7 * n): int(0.85 * n)])
    tr, va = df[df.case.isin(tr_c)], df[df.case.isin(va_c)]
    score = {k: float(np.nanmedian(np.abs(rel_err(va[f"est_{k}"], va)))) + float(np.isnan(rel_err(va[f"est_{k}"], va)).mean()) for k in PHYS_EST}
    base = min(score, key=score.get)
    print("validation score of physics estimators:", {k: round(v, 3) for k, v in score.items()}, "-> base =", base)
else:
    S = pickle.load(open(CLEAN / "model_outputs" / "models_final.pkl", "rb"))
    base = S["base"]


def z_base(d):
    zb = d[f"z_{base}"].copy()
    for k in ["sqrt", "fmb", "slab", "horner"]:
        zb = zb.fillna(d[f"z_{k}"])
    return zb.fillna(0.0)


if RETRAIN:
    grid = [dict(lr=lr, leaves=lv, msl=ms, l2=1.0) for lr in (0.03, 0.08) for lv in (15, 31, 63) for ms in (10, 30)]

    def tune(cols, ytr, score_fn):
        best, bp = None, None
        for g in grid:
            s = float(np.median(np.abs(score_fn(fit(tr[cols], ytr, g)))))
            if best is None or s < best:
                best, bp = s, g
        return bp, best

    t0 = time.time()
    bpD, sD = tune(DATA, tr.z, lambda m: rel_err(va.pwf + (va.pN - va.pwf) * np.exp(m.predict(va[DATA])), va))
    bpP, sP = tune(DATA + PHYS, tr.z - z_base(tr),
                   lambda m: rel_err(va.pwf + (va.pN - va.pwf) * np.exp(z_base(va) + m.predict(va[DATA + PHYS])), va))
    print(f"selected ML-D {bpD} (val {sD:.4f}); ML-P {bpP} (val {sP:.4f}); {time.time() - t0:.0f} s")
else:
    bpD, bpP = S["bpD"], S["bpP"]


def train(d, seed):
    """Point models on all training cases; quantile models on 85 %, conformal calibration on 15 %."""
    rng = np.random.default_rng(seed)
    cs = np.array(sorted(d.case.unique()))
    rng.shuffle(cs)
    cal = set(cs[: int(0.15 * len(cs))])
    fitd, cald = d[~d.case.isin(cal)], d[d.case.isin(cal)]
    mD = fit(d[DATA], d.z, bpD)
    mP = fit(d[DATA + PHYS], d.z - z_base(d), bpP)
    q10 = fit(fitd[DATA + PHYS], fitd.z - z_base(fitd), bpP, "quantile", 0.1)
    q90 = fit(fitd[DATA + PHYS], fitd.z - z_base(fitd), bpP, "quantile", 0.9)
    r = cald.z - z_base(cald)
    sc = np.maximum(q10.predict(cald[DATA + PHYS]) - r, r - q90.predict(cald[DATA + PHYS]))
    Qc = float(np.quantile(sc, min(1.0, 0.8 * (1 + 1 / len(sc)))))
    return dict(mD=mD, mP=mP, mP10=q10, mP90=q90, Qc=Qc, base=base, DATA=DATA, PHYS=PHYS, bpD=bpD, bpP=bpP)


def predict(M, d):
    out = pd.DataFrame(index=d.index)
    for k in EST:
        out[k] = d[f"est_{k}"]
    zb, dN = z_base(d), d.pN - d.pwf
    out["ml_data"] = d.pwf + dN * np.exp(M["mD"].predict(d[DATA]))
    out["ml_phys"] = d.pwf + dN * np.exp(zb + M["mP"].predict(d[DATA + PHYS]))
    out["ml_p10"] = d.pwf + dN * np.exp(zb + M["mP10"].predict(d[DATA + PHYS]) - M["Qc"])
    out["ml_p90"] = d.pwf + dN * np.exp(zb + M["mP90"].predict(d[DATA + PHYS]) + M["Qc"])
    return out


def metrics(d, pr, name, fold=None):
    rows = []
    for N in (24, 72, 168):
        s = d.N == N
        for k in EST + ["ml_data", "ml_phys"]:
            e = rel_err(pr.loc[s, k], d[s])
            a = (pr.loc[s, k] - d.loc[s, "pbar_srv"]).abs()
            rows.append(dict(set=name, fold=fold, N=N, method=k, n=int(s.sum()), fail=float(e.isna().mean()),
                             bias=float(np.nanmedian(e)), mae_rel=float(np.nanmedian(np.abs(e))), mae_psi=float(np.nanmedian(a)),
                             within10=float(np.nanmean(np.abs(e) <= 0.10)), within25=float(np.nanmean(np.abs(e) <= 0.25))))
        cov = ((d.loc[s, "pbar_srv"] >= pr.loc[s, "ml_p10"]) & (d.loc[s, "pbar_srv"] <= pr.loc[s, "ml_p90"])).mean()
        width = np.median((pr.loc[s, "ml_p90"] - pr.loc[s, "ml_p10"]) / (d.loc[s, "pbar_srv"] - d.loc[s, "pwf"]))
        rows.append(dict(set=name, fold=fold, N=N, method="ml_phys_interval", n=int(s.sum()), coverage=float(cov), width_rel=float(width)))
    return rows


KEEP = ["case", "N", "pwf", "pN", "pbar_srv", "pbar_el", "dtD", "tpD", "par_L", "par_k", "par_xf", "par_Ye", "par_Fc", "par_C_well"]
if RETRAIN:
    t0 = time.time()
    cases = np.array(sorted(df.case.unique()))
    rng = np.random.default_rng(7)
    rng.shuffle(cases)
    rows, oof = [], []
    for f, test_cases in enumerate(np.array_split(cases, 5)):
        te = df[df.case.isin(test_cases)]
        M = train(df[~df.case.isin(test_cases)], 100 + f)
        pr = predict(M, te)
        rows += metrics(te, pr, "cv", f)
        oof.append(pd.concat([te[KEEP], pr], axis=1).assign(set="cv", fold=f))
    OOF = pd.concat(oof)
    rows += metrics(OOF, OOF, "cv_pooled")
    Mfin = train(df, 999)
    preds = [OOF]
    for name in ("nonlinear", "extrap"):
        od = TABLES[name]
        od = od[np.isfinite(od.z)].reset_index(drop=True)
        po = predict(Mfin, od)
        rows += metrics(od, po, name)
        preds.append(pd.concat([od[KEEP], po], axis=1).assign(set=name, fold=-1))
    METRICS = pd.DataFrame(rows)
    PRED = pd.concat(preds)
    pickle.dump(Mfin, open(OUT / "models_final_rerun.pkl", "wb"))
    METRICS.to_csv(OUT / "metrics_all_rerun.csv", index=False)
    print(f"cross-validation and final models: {time.time() - t0:.0f} s")
else:
    Mfin = S
    METRICS = rcsv(CLEAN / "model_outputs" / "metrics_all.csv")
    PRED = rcsv(CLEAN / "model_outputs" / "predictions_cv_and_ood.csv")

validation score of physics estimators: {'horner': 0.362, 'fmb': 0.454, 'sqrt': 0.427, 'slab': 0.595} -> base = horner


selected ML-D {'lr': 0.08, 'leaves': 63, 'msl': 10, 'l2': 1.0} (val 0.1497); ML-P {'lr': 0.03, 'leaves': 31, 'msl': 30, 'l2': 1.0} (val 0.1427); 174 s


cross-validation and final models: 202 s


In [10]:
# Table 2 (pooled out-of-fold predictions) and fold-to-fold spread
pool = METRICS[METRICS.set == "cv_pooled"]
spread = METRICS[METRICS.set == "cv"].groupby(["N", "method"]).mae_rel.std().rename("mae_rel_fold_sd").reset_index()
tab2 = pool.merge(spread, on=["N", "method"], how="left")
NAMES = {"proxy": "Proxy", "horner": "Horner", "fmb": "FMB", "sqrt": "Square-root decline", "slab": "Slab inversion",
         "ml_data": "ML-D", "ml_phys": "ML-P"}
t2 = []
for m in EST + ["ml_data", "ml_phys"]:
    r = {"Estimator": NAMES[m]}
    for N in (24, 72, 168):
        x = tab2[(tab2.method == m) & (tab2.N == N)].iloc[0]
        r[f"N={N} h"] = f"{x.mae_rel:.3f} ({x.mae_psi:.0f})"
    x = tab2[(tab2.method == m) & (tab2.N == 72)].iloc[0]
    r["Bias, 72 h"], r["Within 25 %, 72 h"] = f"{x.bias:+.3f}", f"{x.within25:.2f}"
    t2.append(r)
r = {"Estimator": "ML-P 80 % interval: coverage (width/deficit)"}
for N in (24, 72, 168):
    x = tab2[(tab2.method == "ml_phys_interval") & (tab2.N == N)].iloc[0]
    r[f"N={N} h"] = f"{x.coverage:.3f} ({x.width_rel:.2f})"
t2.append(r)
TABLE2 = pd.DataFrame(t2)
print("Table 2"); print(TABLE2.to_string(index=False))
print("ML-P fold-to-fold s.d. of median |e|:", tab2[tab2.method == "ml_phys"].set_index("N").mae_rel_fold_sd.round(3).to_dict())
# Table 3
t3 = []
for m in EST + ["ml_data", "ml_phys"]:
    r = {"Estimator": NAMES[m]}
    for sname, lab in (("nonlinear", "Nonlinear"), ("extrap", "Extrapolation")):
        x = METRICS[(METRICS.set == sname) & (METRICS.N == 72) & (METRICS.method == m)].iloc[0]
        r[f"{lab}: median |e|"], r[f"{lab}: bias"] = f"{x.mae_rel:.3f}", f"{x.bias:+.3f}"
    t3.append(r)
r = {"Estimator": "ML-P 80 % interval coverage"}
for sname, lab in (("nonlinear", "Nonlinear"), ("extrap", "Extrapolation")):
    r[f"{lab}: median |e|"] = f"{METRICS[(METRICS.set == sname) & (METRICS.N == 72) & (METRICS.method == 'ml_phys_interval')].coverage.iloc[0]:.2f}"
t3.append(r)
TABLE3 = pd.DataFrame(t3).fillna("—")
print("\nTable 3"); print(TABLE3.to_string(index=False))
TABLE2.to_csv(OUT / "Table2_rerun.csv", index=False); TABLE3.to_csv(OUT / "Table3_rerun.csv", index=False)

Table 2
                                   Estimator       N=24 h       N=72 h      N=168 h Bias, 72 h Within 25 %, 72 h
                                       Proxy  0.712 (858)  0.584 (695)  0.473 (531)     -0.579              0.13
                                      Horner  0.430 (345)  0.376 (212)  0.366 (215)     -0.115              0.34
                                         FMB  0.406 (314)  0.409 (316)  0.412 (311)     -0.234              0.28
                         Square-root decline  0.466 (367)  0.365 (278)  0.331 (257)     +0.023              0.34
                              Slab inversion  0.659 (560)  0.533 (414)  0.454 (334)     +0.377              0.24
                                        ML-D  0.175 (150)  0.157 (128)  0.147 (119)     +0.009              0.70
                                        ML-P  0.159 (129)  0.144 (120)  0.139 (112)     +0.003              0.74
ML-P 80 % interval: coverage (width/deficit) 0.800 (0.67) 0.801 (0.62) 0.819 (0.61)     

In [11]:
# Sensitivity to a 15 % initial-pressure error on the held-out cases of the selection split (Section 6.4)
if RETRAIN:
    rs = np.random.default_rng(42); cs_ = np.array(sorted(df.case.unique())); rs.shuffle(cs_)
    n_ = len(cs_); tr_s, va_s, te_s = set(cs_[: int(0.7 * n_)]), set(cs_[int(0.7 * n_): int(0.85 * n_)]), set(cs_[int(0.85 * n_):])
    trva = df[~df.case.isin(te_s)]
    mD_s, mP_s = fit(trva[DATA], trva.z, bpD), fit(trva[DATA + PHYS], trva.z - z_base(trva), bpP)
    for name in ("main", "main_pierr15"):
        d_ = TABLES[name]; d_ = d_[np.isfinite(d_.z) & d_.case.isin(te_s) & (d_.N == 72)]
        pP = d_.pwf + (d_.pN - d_.pwf) * np.exp(z_base(d_) + mP_s.predict(d_[DATA + PHYS]))
        print(f"{name:13s}: ML-P 72-h median |e| = {np.median(np.abs(rel_err(pP, d_))):.3f}, FMB = {np.nanmedian(np.abs(rel_err(d_.est_fmb, d_))):.3f}")
# Error by regime (Section 6.3)
cv = PRED[PRED.set == "cv"]
s = cv[cv.N == 72]
e = np.abs(rel_err(s.ml_phys, s))
for lo, hi in ((0, 1e-3), (1e-3, 0.03), (0.03, 0.2), (0.2, 1e9)):
    m = (s.dtD >= lo) & (s.dtD < hi)
    print(f"ML-P, N = 72 h, dt_D in [{lo:g}, {hi:g}): median |e| = {e[m].median():.3f} (n = {m.sum()})")
for lo, hi in ((0, 0.3), (0.3, 1e9)):
    m = (s.tpD >= lo) & (s.tpD < hi)
    print(f"square-root decline, t_pD in [{lo}, {hi:g}): bias {np.nanmedian(rel_err(s.sqrt[m], s[m])):+.3f}")
for lo, hi in ((0, 3), (3, 10), (10, 30), (30, 100), (100, 1e9)):
    sel = s[(s.tpD >= 0.3)]
    fcd = sel.par_Fc / (sel.par_k * sel.par_xf)
    m = (fcd > lo) & (fcd <= hi)
    print(f"square-root decline, t_pD >= 0.3, F_cD in ({lo}, {hi:g}]: bias {np.nanmedian(rel_err(sel.sqrt[m], sel[m])):+.3f} (n = {m.sum()})")
ratio = (s.sqrt - s.pwf) / (s.pbar_srv - s.pwf)
ratio_p = (s.ml_phys - s.pwf) / (s.pbar_srv - s.pwf)
print("central 80 % of estimated/true deficit: square-root", np.nanpercentile(ratio, [10, 90]).round(2), "| ML-P", np.nanpercentile(ratio_p, [10, 90]).round(2))

main         : ML-P 72-h median |e| = 0.145, FMB = 0.425
main_pierr15 : ML-P 72-h median |e| = 0.144, FMB = 0.452
ML-P, N = 72 h, dt_D in [0, 0.001): median |e| = 0.257 (n = 260)
ML-P, N = 72 h, dt_D in [0.001, 0.03): median |e| = 0.136 (n = 1777)
ML-P, N = 72 h, dt_D in [0.03, 0.2): median |e| = 0.123 (n = 745)
ML-P, N = 72 h, dt_D in [0.2, 1e+09): median |e| = 0.151 (n = 199)
square-root decline, t_pD in [0, 0.3): bias -0.336
square-root decline, t_pD in [0.3, 1e+09): bias +0.157
square-root decline, t_pD >= 0.3, F_cD in (0, 3]: bias +0.308 (n = 167)
square-root decline, t_pD >= 0.3, F_cD in (3, 10]: bias +0.381 (n = 361)
square-root decline, t_pD >= 0.3, F_cD in (10, 30]: bias +0.258 (n = 460)
square-root decline, t_pD >= 0.3, F_cD in (30, 100]: bias +0.097 (n = 546)
square-root decline, t_pD >= 0.3, F_cD in (100, 1e+09]: bias +0.047 (n = 570)
central 80 % of estimated/true deficit: square-root [0.45 1.97] | ML-P [0.74 1.35]


## 6. Application to HFTS-1 (Section 7, Table 4)

### 6.1 Pad production from the RRC lease (raw) and shut-in detection from the raw 10–60 s gauge records

In [12]:
lease = rcsv(RAW / "field" / "RRC_lease_16507_monthly_production_raw.csv")
lease["date"] = pd.to_datetime(lease.month, format="%b %Y")
lease["days"] = lease.date.dt.days_in_month
lease["q_lease"] = lease.oil_bbl / lease.days
bw = lease[(lease.date >= "2015-04-01") & (lease.date <= "2015-09-01")]
tb = (bw.date - pd.Timestamp("2015-01-01")).dt.days.values
slope, icpt = np.polyfit(tb, np.log(bw.q_lease.values), 1)
pad = lease[(lease.date >= "2016-01-01") & (lease.date <= "2016-09-01")].copy()
tt = (pad.date - pd.Timestamp("2015-01-01")).dt.days.values + 15
for name, s_ in (("base_mid", slope), ("base_slow", slope * 0.5), ("base_fast", slope * 1.5)):
    pad[f"q_pad_{name}"] = pad.q_lease - np.exp(icpt + slope * tb.mean()) * np.exp(s_ * (tt - tb.mean()))
print(f"older-well base decline {-slope * 365:.2f} 1/yr")
print(pad[["month", "q_lease", "q_pad_base_mid", "q_pad_base_slow", "q_pad_base_fast"]].round(0).to_string(index=False))

raw = rcsv(RAW / "field" / "HFTS1_BHP_raw_2016-07-31_to_2016-08-16.csv", parse_dates=["time"])
win = rcsv(RAW / "field" / "HFTS1_aug2016_buildup_windows_approx.csv", parse_dates=["shut_in", "reopened"])
CUT = {"4SM": pd.Timestamp("2016-08-08 12:00")}
PRE_H = {"4SM": 0.4}
detected = []
for _, r in win.iterrows():
    w = r.well
    s_ = raw[(raw.well == w) & (raw.p_psia > 0) & (raw.p_psia < 6000)].set_index("time").p_psia.sort_index()
    seg0 = s_[r.shut_in - pd.Timedelta(hours=PRE_H.get(w, 3.0)): r.reopened + pd.Timedelta(hours=1)]
    m1 = seg0.resample("1min").median().dropna()
    dd = m1.diff().rolling(5).sum()
    i0 = None
    for cand in dd[dd > 10.0].index:      # sustained rise: > 20 psi above the flowing level 25-35 min later
        ref = m1[cand - pd.Timedelta(minutes=70): cand - pd.Timedelta(minutes=10)]
        later = m1[cand + pd.Timedelta(minutes=25): cand + pd.Timedelta(minutes=35)]
        if len(ref) > 10 and len(later) and later.median() > ref.median() + 20.0:
            i0 = cand
            break
    base_ = m1[: i0 - pd.Timedelta(minutes=6)]
    pwf = float(base_.tail(60).median())
    sd = float(base_.tail(60).std())
    pre = m1[i0 - pd.Timedelta(minutes=10): i0]
    flowing = pre[pre <= pwf + max(2.0, 2 * sd)]
    t0_ = flowing.index[-1] if len(flowing) else i0 - pd.Timedelta(minutes=5)
    after = m1[t0_:]
    d2 = after.diff().rolling(10).sum()
    reo = d2[d2 < -40].index
    t1_ = after[:reo[0]].idxmax() if len(reo) else after.index[-1]
    if w in CUT:
        t1_ = min(t1_, CUT[w])
    detected.append(dict(well=w, shut_in=t0_, reopened_or_truncated=t1_, duration_h=(t1_ - t0_).total_seconds() / 3600, pwf=pwf))
DET = pd.DataFrame(detected)
SUM = rcsv(CLEAN / "field" / "HFTS1_buildups_summary.csv", parse_dates=["shut_in", "reopened_or_truncated", "production_start"])
chk = DET.merge(SUM, on="well", suffixes=("_raw", "_clean"))
print("\nshut-in detection from raw gauges vs packaged clean summary")
print(chk[["well", "shut_in_raw", "shut_in_clean", "duration_h_raw", "duration_h_clean", "pwf", "pwf_last_flowing_psia"]].round(2).to_string(index=False))

older-well base decline 0.69 1/yr


   month  q_lease  q_pad_base_mid  q_pad_base_slow  q_pad_base_fast
Jan 2016   7512.0          5546.0           5104.0           5908.0
Feb 2016   7615.0          5762.0           5277.0           6146.0
Mar 2016   5952.0          4198.0           3677.0           4600.0
Apr 2016   4864.0          3210.0           2655.0           3626.0
May 2016   4331.0          2769.0           2184.0           3194.0
Jun 2016   4500.0          3027.0           2415.0           3459.0
Jul 2016   4156.0          2765.0           2130.0           3201.0
Aug 2016   3672.0          2360.0           1705.0           2797.0
Sep 2016   3392.0          2156.0           1482.0           2592.0



shut-in detection from raw gauges vs packaged clean summary
well         shut_in_raw       shut_in_clean  duration_h_raw  duration_h_clean     pwf  pwf_last_flowing_psia
 3SU 2016-08-04 11:09:00 2016-08-04 11:09:00           36.05             36.05  767.17                 767.17
 5SM 2016-08-06 11:13:00 2016-08-06 11:13:00           71.92             71.92  981.78                 981.78
 5SU 2016-08-07 00:00:00 2016-08-07 00:00:00           71.57             71.57  798.56                 798.56
 6SU 2016-08-07 11:57:00 2016-08-07 11:57:00           71.12             71.12  621.84                 621.83
 6SM 2016-08-07 23:57:00 2016-08-07 23:57:00           71.47             71.47  760.30                 760.30
 7RM 2016-08-08 11:16:00 2016-08-08 11:16:00           77.83             77.83  759.38                 759.39
 8SM 2016-08-09 13:24:00 2016-08-09 13:24:00           69.90             69.90  726.19                 726.19
 7SU 2016-08-09 23:21:00 2016-08-09 23:21:00           59.8

### 6.2 Monte Carlo field estimates (Table 4)

Unknowns propagated over 300 draws: initial pressure at gauge depth (uniform 3,700–4,800 psia), base-decline variant (slow/mid/fast) and 5 % rate noise.
Wells are processed in the original run order so that the random draws, and hence Table 4, are reproduced exactly.

In [13]:
CUR = rcsv(CLEAN / "field" / "HFTS1_buildup_curves.csv")
HIS = rcsv(CLEAN / "field" / "HFTS1_flowing_pressure_histories.csv")
HOUR = rcsv(CLEAN / "field" / "HFTS1_hourly_median_BHP.csv", parse_dates=["time"]).set_index("time")
PADC = rcsv(CLEAN / "field" / "HFTS1_pad_production_monthly.csv", parse_dates=["date"])
M = Mfin
rng = np.random.default_rng(2016)
NMC = 300
rows = []
for _, b in SUM.sort_values("order").iterrows():
    w = b.well
    bu_t, bu_p = CUR[CUR.well == w].dt_h.values, CUR[CUR.well == w].pws_psia.values
    # gauge pressures were recorded in single precision; keep them so to reproduce the paper's numbers exactly
    hist_t, hist_p = HIS[HIS.well == w].t_days.values, HIS[HIS.well == w].pwf_psia.values.astype(np.float32)
    for N in (24.0, min(72.0, float(b.duration_h))):
        ests = []
        for k in range(NMC):
            variant = ["base_slow", "base_mid", "base_fast"][k % 3]
            ps = b.production_start
            t_shut = (b.shut_in - ps).total_seconds() / 86400
            qm, tm_end, pwm, dur = [], [], [], []
            for _, r in PADC.iterrows():
                m0 = (r.date - ps).total_seconds() / 86400
                a, e = max(m0, 0.0), min(m0 + r.days, t_shut)
                if e - a < 5:
                    continue
                qm.append(max(r[f"q_pad_{variant}"], 1.0) * (1 + 0.05 * rng.standard_normal()))
                tm_end.append(e)
                sel = (hist_t >= a) & (hist_t < e)
                pwm.append(float(np.mean(hist_p[sel])) if sel.any() else np.nan)
                dur.append(e - a)
            qm, tm_end, pwm, dur = map(np.array, (qm, tm_end, pwm, dur))
            ok = np.isfinite(pwm)
            qm, tm_end, pwm, dur = qm[ok], tm_end[ok], pwm[ok], dur[ok]
            Qm = np.cumsum(qm * dur)
            o = dict(tp=t_shut, pwf_end=b.pwf_last_flowing_psia, pi_obs=rng.uniform(3700.0, 4800.0), qm=qm, Qm=Qm,
                     tm_end=tm_end, tm_mid=tm_end - dur / 2, pwm=pwm, tdays=hist_t, pwd=hist_p,
                     bu_t=bu_t[bu_t <= N + 1e-9], bu_p=bu_p[bu_t <= N + 1e-9], N=N, Q_total=float(Qm[-1]))
            d, ph, e, pwf, pN = features(o)
            X = pd.DataFrame([{**d, **ph}])
            zb = ph[f"z_{M['base']}"]
            if not np.isfinite(zb):
                zb = next((ph[f"z_{kk}"] for kk in ["sqrt", "fmb", "slab", "horner"] if np.isfinite(ph[f"z_{kk}"])), 0.0)
            dN = pN - pwf
            e["ml_data"] = pwf + dN * float(np.exp(M["mD"].predict(X[M["DATA"]])[0]))
            e["ml_phys"] = pwf + dN * float(np.exp(zb + M["mP"].predict(X[M["DATA"] + M["PHYS"]])[0]))
            e["ml_p10"] = pwf + dN * float(np.exp(zb + M["mP10"].predict(X[M["DATA"] + M["PHYS"]])[0] - M["Qc"]))
            e["ml_p90"] = pwf + dN * float(np.exp(zb + M["mP90"].predict(X[M["DATA"] + M["PHYS"]])[0] + M["Qc"]))
            ests.append(e)
        E = pd.DataFrame(ests)
        row = dict(well=w, N=N, dur_h=b.duration_h, tp_days=b.tp_days, pwf=b.pwf_last_flowing_psia,
                   pN=float(np.interp(N, bu_t, bu_p)), p_core_well=b.p_core_well_6TW_psia)
        for kx in ["proxy", "horner", "fmb", "sqrt", "slab", "ml_data", "ml_phys"]:
            v = E[kx].astype(float)
            row[f"{kx}_p50"], row[f"{kx}_p10"], row[f"{kx}_p90"] = float(np.nanmedian(v)), float(np.nanpercentile(v, 10)), float(np.nanpercentile(v, 90))
        lo_q, hi_q = float(np.nanpercentile(E["ml_p10"], 10)), float(np.nanpercentile(E["ml_p90"], 90))
        row.update(ml_phys_lo_q=lo_q, ml_phys_hi_q=hi_q, quantile_inconsistent=bool(row["ml_phys_p50"] < lo_q or row["ml_phys_p50"] > hi_q),
                   ml_phys_lo=min(lo_q, row["ml_phys_p50"]), ml_phys_hi=max(hi_q, row["ml_phys_p50"]), D_p50=float(np.nanmedian(E["D"])))
        rows.append(row)
FIELD = pd.DataFrame(rows)
FIELD.to_csv(OUT / "HFTS1_field_estimates_rerun.csv", index=False)
F = FIELD.loc[FIELD.groupby("well").N.idxmax()].sort_values("well").reset_index(drop=True)
TABLE4 = pd.DataFrame(dict(Well=F.well, tp_d=F.tp_days.round(0).astype(int), N_h=F.N.round(0).astype(int), pwf=F.pwf.round(0).astype(int),
                           pws_N=F.pN.round(0).astype(int), Horner=F.horner_p50.round(0).astype(int), Sqrt_root=F.sqrt_p50.round(0).astype(int),
                           ML_D=F.ml_data_p50.round(0).astype(int), ML_P=F.ml_phys_p50.round(0).astype(int),
                           interval=[f"{a:,.0f}–{b:,.0f}" for a, b in zip(F.ml_phys_lo, F.ml_phys_hi)],
                           core_6TW=F.p_core_well.round(0).astype(int), D_1e3_per_day=(1000 * F.D_p50).round(1),
                           flagged=F.quantile_inconsistent))
print("Table 4"); print(TABLE4.to_string(index=False))
TABLE4.to_csv(OUT / "Table4_rerun.csv", index=False)
F9 = F[F.N > 50]
tauL = np.pi**2 / (4 * F9.D_p50)
print(f"\nnine wells: ML-P {F9.ml_phys_p50.min():.0f}-{F9.ml_phys_p50.max():.0f} psia; buildup end {F9.pN.min():.0f}-{F9.pN.max():.0f}; "
      f"recovered {((F9.pN - F9.pwf) / (F9.ml_phys_p50 - F9.pwf)).min():.2f}-{((F9.pN - F9.pwf) / (F9.ml_phys_p50 - F9.pwf)).max():.2f} of the estimated deficit")
print(f"tau_L {tauL.min():.0f}-{tauL.max():.0f} d; L {np.sqrt(eta_hfts * 24 * tauL).min():.0f}-{np.sqrt(eta_hfts * 24 * tauL).max():.0f} ft; "
      f"t_pD {(F9.tp_days / tauL).min():.2f}-{(F9.tp_days / tauL).max():.2f}")
paper = rcsv(CLEAN / "field" / "HFTS1_field_estimates.csv")
cmp_ = FIELD.merge(paper, on=["well", "N"], suffixes=("", "_paper"))
print("max |ML-P median difference| vs the packaged field estimates:", round(float(np.max(np.abs(cmp_.ml_phys_p50 - cmp_.ml_phys_p50_paper))), 3), "psi")

Table 4
Well  tp_d  N_h  pwf  pws_N  Horner  Sqrt_root  ML_D  ML_P    interval  core_6TW  D_1e3_per_day  flagged
 3SU   217   36  767   1090    1495       2809  2620  3198 2,020–3,796      2848            3.7    False
 4SM   208   72 1071   1259    1487       1854  1411  2267 1,540–2,267      2844            4.3     True
 5SM   204   72  982   1274    1605       1912  1486  1970 1,442–2,137      2839            6.8    False
 5SU   215   72  799   1193    1630       1908  1478  1704 1,402–2,049      2836            8.0    False
 6SM   207   71  760   1170    1627       1748  1438  1580 1,352–2,026      2830           10.4    False
 6SU   219   71  622   1082    1574       1791  1299  1489 1,265–2,089      2834           10.6    False
 7RM   189   72  759   1114    1560       2256  1379  2372 1,473–2,480      2828            4.4    False
 7SU   229   60  612   1016    1417       1836  1282  1656 1,213–1,971      2826            6.4    False
 8SM   222   70  726   1001    1272       1559 

## 7. Manuscript figures (written to `05_Code/outputs/figures`)

In [14]:
BLUE, ORANGE, AQUA, INK, INK2, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#e4e3df"
SEQ = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "savefig.dpi": 300, "savefig.bbox": "tight"})
from matplotlib.patches import Rectangle

# Fig. 1: element geometry
fig, ax = plt.subplots(figsize=(6.5, 3.4)); ax.grid(False)
L_, xf_, Ye_ = 1.3, 2.0, 3.0
for i in range(-3, 3):
    x0 = 2 * L_ * i
    ax.plot([x0, x0], [-xf_, xf_], color=ORANGE, lw=2.2, solid_capstyle="round")
    ax.plot([x0 + L_] * 2, [-Ye_, Ye_], color=INK2, lw=0.6, ls=(0, (3, 3)))
ax.plot([-8, 6.5], [0, 0], color=INK, lw=2.6)
ax.plot([-8, 6.5], [2 * Ye_, 2 * Ye_], color=INK, lw=1.4, alpha=0.4)
for yy in (Ye_, -Ye_):
    ax.plot([-8, 6.5], [yy, yy], color=INK2, lw=0.6, ls=(0, (3, 3)))
ax.add_patch(Rectangle((0, xf_), L_, Ye_ - xf_, facecolor=BLUE, alpha=0.10, edgecolor="none"))
ax.add_patch(Rectangle((0, 0), L_, xf_, facecolor=BLUE, alpha=0.30, edgecolor="none"))
ax.add_patch(Rectangle((0, 0), L_, Ye_, facecolor="none", edgecolor=BLUE, lw=1.6))
ax.text(L_ / 2, Ye_ + 0.18, "element $\\Omega$ (closed)", color=BLUE, ha="center", va="bottom", fontsize=8.5)
ax.annotate("stimulated slab $\\Omega_S$:  $0<y<x_f$", xy=(L_ * 0.6, xf_ * 0.5), xytext=(7.0, 1.0), color=BLUE, fontsize=8,
            arrowprops=dict(arrowstyle="-", color=BLUE, lw=0.7), va="center")
ax.annotate("outer region:  $x_f<y<Y_e$", xy=(L_ * 0.6, (xf_ + Ye_) / 2), xytext=(7.0, 2.6), color=INK2, fontsize=8,
            arrowprops=dict(arrowstyle="-", color=INK2, lw=0.7), va="center")
ax.annotate("", xy=(0, -0.55), xytext=(L_, -0.55), arrowprops=dict(arrowstyle="<->", color=INK2, lw=0.8, shrinkA=0, shrinkB=0))
ax.text(L_ / 2, -0.75, "$L$", ha="center", va="top", color=INK2)
ax.annotate("", xy=(-3.3, 0), xytext=(-3.3, xf_), arrowprops=dict(arrowstyle="<->", color=INK2, lw=0.8, shrinkA=0, shrinkB=0))
ax.text(-3.45, xf_ / 2, "$x_f$", ha="right", va="center", color=INK2)
ax.annotate("", xy=(-5.9, 0), xytext=(-5.9, Ye_), arrowprops=dict(arrowstyle="<->", color=INK2, lw=0.8, shrinkA=0, shrinkB=0))
ax.text(-6.05, Ye_ / 2, "$Y_e$", ha="right", va="center", color=INK2)
ax.text(7.0, 0.0, "wellbore ($y=0$)", color=INK, fontsize=8, va="center")
ax.text(7.0, 2 * Ye_, "neighbouring well", color=INK2, fontsize=8, va="center")
ax.text(7.0, -Ye_, "no-flow symmetry planes (dashed)", color=INK2, fontsize=8, va="center")
ax.text(7.0, -xf_, "transverse fractures", color=ORANGE, fontsize=8, va="center")
ax.plot([6.5, 6.9], [-xf_, -xf_], color=ORANGE, lw=0.7)
ax.set_xlim(-8, 11.5); ax.set_ylim(-Ye_ - 0.2, 2 * Ye_ + 0.3); ax.set_aspect("equal"); ax.axis("off")
fig.savefig(FIGS / "Fig01_element_geometry.png"); plt.close(fig)

# Fig. 2: recovery law and shut-in time needed
dt = np.geomspace(1e-5, 1.0, 300)
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8))
ax = axes[0]
for c, tp in zip(SEQ, [0.003, 0.01, 0.03, 0.1, 0.3]):
    ax.plot(dt, recovered_fraction(dt, tp), color=c, lw=1.6, label=f"$t_{{pD}}={tp:g}$")
ax.plot(dt, recovered_fraction_bdf(dt), color=INK, lw=1.2, ls=(0, (4, 2)), label="$t_{pD}\\to\\infty$, Eq. (10)")
ax.plot(dt[dt < 0.08], (np.pi**1.5 / 2) * np.sqrt(dt[dt < 0.08]), color=ORANGE, lw=1.0, ls=":", label="$\\frac{\\pi^{3/2}}{2}\\sqrt{\\Delta t_D}$")
ax.set_xscale("log"); ax.set_xlabel("$\\Delta t_D = \\eta\\,\\Delta t/L^2$"); ax.set_ylabel("recovered fraction $R$"); ax.set_ylim(0, 1.02)
ax.legend(fontsize=6.8, loc="upper left", bbox_to_anchor=(0.0, 0.93), handlelength=1.8)
ax.text(0.02, 0.97, "(a)", transform=ax.transAxes, va="top", ha="left", fontweight="bold")
ax = axes[1]
Ls = np.linspace(15, 150, 200)
for c, eh, lab in [(SEQ[1], 0.15, "$\\eta=0.15$ ft$^2$/h"), (SEQ[3], 0.43, "$\\eta=0.43$ ft$^2$/h (HFTS-1 RTA)"), (SEQ[5], 1.2, "$\\eta=1.2$ ft$^2$/h")]:
    ax.plot(Ls, 0.1923 * Ls**2 / eh / 24, color=c, lw=1.6, label=lab)
ax.axhline(3, color=ORANGE, lw=1.0, ls=(0, (4, 2)))
ax.text(60, 2.6, "72-h test (HFTS-1, Aug 2016)", color=ORANGE, ha="left", va="top", fontsize=7.5)
ax.set_yscale("log"); ax.set_xlabel("effective half fracture spacing $L$ (ft)"); ax.set_ylabel("shut-in for $R=0.9$ (days)")
ax.legend(fontsize=7, loc="lower right", bbox_to_anchor=(1.0, 0.12))
ax.text(0.02, 0.97, "(b)", transform=ax.transAxes, va="top", ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIGS / "Fig02_recovery_law.png"); plt.close(fig)

# Fig. 3: verification
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8))
ax = axes[0]
for c, (Lv, tp) in zip([BLUE, AQUA, ORANGE], [(30.0, 60.0), (45.0, 150.0), (60.0, 400.0)]):
    el = Element(L=Lv, xf=200.0, Ye=200.0, h=100.0, k=k_, phi=phi_, mu=mu_, ct=ct_, Fc=1e5, Cw=0.0, nx_r=1.12, ny_r=1.3)
    r = el.run(5000.0, lambda t: 1000.0, tp, sh, dt_max=0.1)
    Le = Lv - 0.01
    tpD = eta_d * tp / Le**2
    dtD = eta_d * sh / 24 / Le**2
    ax.plot(dtD, recovered_fraction(dtD, tpD), color=c, lw=1.4, label=f"Eq. (9), $t_{{pD}}={tpD:.2f}$")
    ax.plot(dtD, (r["pws"] - 1000.0) / (r["pbar_srv"] - 1000.0), "o", color=c, ms=3.2, mfc="white", mew=1.0)
ax.set_xscale("log"); ax.set_xlabel("$\\Delta t_D$"); ax.set_ylabel("$R$ (lines: closed form; circles: simulator)")
ax.legend(fontsize=7, loc="upper left"); ax.text(0.97, 0.05, "(a)", transform=ax.transAxes, ha="right", fontweight="bold")
ax = axes[1]
sh2 = np.geomspace(0.05, 504, 40)
for c, (nxr, nyr, lab) in zip([SEQ[1], SEQ[3], SEQ[5]], [(1.4, 1.5, "coarse"), (1.25, 1.3, "base"), (1.12, 1.15, "fine")]):
    el = Element(L=45.0, xf=150.0, Ye=330.0, h=100.0, k=k_, phi=phi_, mu=mu_, ct=ct_, Fc=5.0, Cw=0.002, nx_r=nxr, ny_r=nyr)
    r = el.run(5000.0, pwf_f, 220.0, sh2)
    ax.plot(sh2, r["pws"], color=c, lw=1.4, label=f"{lab} ({el.n} cells), $\\bar p_S$={r['pbar_srv']:.0f} psia")
ax.axhline(r["pbar_srv"], color=INK, lw=0.9, ls=(0, (4, 2)))
ax.text(0.06, r["pbar_srv"] + 15, "$\\bar p_S$ (stimulated-region average)", fontsize=7, color=INK)
ax.set_xscale("log"); ax.set_xlabel("shut-in time $\\Delta t$ (h)"); ax.set_ylabel("$p_{ws}$ (psia)")
ax.legend(fontsize=7, loc="center left"); ax.text(0.97, 0.05, "(b)", transform=ax.transAxes, ha="right", fontweight="bold")
fig.tight_layout(); fig.savefig(FIGS / "Fig03_verification.png"); plt.close(fig)

# Fig. 4: proxy bias against theory
T = TABLES["main"]
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8))
ax = axes[0]
R_ = (T.pN - T.pwf) / (T.pbar_srv - T.pwf)
ok = np.isfinite(R_) & (T.pbar_srv - T.pwf > 20)
ax.scatter(T.dtD[ok], R_[ok], s=3, color=BLUE, alpha=0.25, lw=0, label="simulated (all cases, N = 24/72/168 h)")
x = np.geomspace(1e-5, 3, 300)
ax.plot(x, recovered_fraction_bdf(x), color=INK, lw=1.4, label="Eq. (10), boundary-dominated slab")
ax.plot(x, recovered_fraction(x, 0.01), color=ORANGE, lw=1.2, ls=(0, (4, 2)), label="Eq. (9), $t_{pD}=0.01$")
ax.set_xscale("log"); ax.set_xlim(1e-5, 3); ax.set_ylim(-0.05, 1.6)
ax.set_xlabel("$\\Delta t_D=\\eta N/L^2$"); ax.set_ylabel("$R_N=(p_{ws}(N)-p_{wf})/(\\bar p_S-p_{wf})$")
ax.legend(fontsize=6.5, loc="upper left"); ax.text(0.97, 0.04, "(a)", transform=ax.transAxes, ha="right", fontweight="bold")
ax = axes[1]
for c, N in zip([BLUE, ORANGE, AQUA], (24, 72, 168)):
    s_ = T[(T.N == N) & ok]
    e_ = np.sort(((s_.pN - s_.pbar_srv) / (s_.pbar_srv - s_.pwf)).values)
    ax.plot(e_, np.linspace(0, 1, len(e_)), color=c, lw=1.6, label=f"N = {N} h")
ax.axvline(0, color=INK2, lw=0.8); ax.set_xlim(-1.05, 0.6)
ax.set_xlabel("proxy error $(p_{ws}(N)-\\bar p_S)/(\\bar p_S-p_{wf})$"); ax.set_ylabel("cumulative fraction of cases")
ax.legend(fontsize=7, loc="lower right", bbox_to_anchor=(1.0, 0.12)); ax.text(0.97, 0.04, "(b)", transform=ax.transAxes, ha="right", fontweight="bold")
fig.tight_layout(); fig.savefig(FIGS / "Fig04_proxy_bias.png"); plt.close(fig)

# Fig. 5: accuracy by estimator and duration
te = PRED[PRED.set == "cv"]
methods = EST + ["ml_data", "ml_phys"]
LONG = {"proxy": "N-hour proxy", "horner": "Horner p*", "fmb": "Flowing MB", "sqrt": "Square-root decline (14)",
        "slab": "Slab inversion", "ml_data": "ML-D (data only)", "ml_phys": "ML-P (physics-constrained)"}
fig, ax = plt.subplots(figsize=(6.5, 3.0))
for j, (c, N) in enumerate(zip([BLUE, ORANGE, AQUA], (24, 72, 168))):
    s_ = te[te.N == N]
    for i, m in enumerate(methods):
        e_ = np.abs((s_[m] - s_.pbar_srv) / (s_.pbar_srv - s_.pwf)).dropna()
        y = i + (j - 1) * 0.22
        q25, q50, q75 = np.percentile(e_, [25, 50, 75])
        ax.plot([q25, q75], [y, y], color=c, lw=2.2, solid_capstyle="round", alpha=0.55)
        ax.plot(q50, y, "o", color=c, ms=5, mec="white", mew=0.8, label=f"N = {N} h" if i == 0 else None)
ax.set_yticks(range(len(methods))); ax.set_yticklabels([LONG[m] for m in methods]); ax.invert_yaxis()
ax.set_xlabel("|relative error of the deficit| (cross-validated, all cases): median (dot) and interquartile range")
ax.set_xlim(0, 1.0); ax.legend(fontsize=7, loc="lower right")
fig.tight_layout(); fig.savefig(FIGS / "Fig05_estimator_accuracy.png"); plt.close(fig)

# Fig. 6: parity at 72 h
s_ = te[te.N == 72]
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.0))
for ax, m, c, lab in [(axes[0], "sqrt", ORANGE, "(a)"), (axes[1], "ml_phys", BLUE, "(b)")]:
    t_ = s_.pbar_srv - s_.pwf
    ax.scatter(t_, s_[m] - s_.pwf, s=6, color=c, alpha=0.5, lw=0)
    ax.scatter(t_, s_.pN - s_.pwf, s=4, color=INK2, alpha=0.25, lw=0, label="72-h proxy")
    lim = [max(1, t_.min() * 0.8), t_.max() * 1.3]
    ax.plot(lim, lim, color=INK, lw=0.9)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("true deficit $\\bar p_S-p_{wf}$ (psi)"); ax.set_ylabel("estimated deficit (psi)")
    ax.set_title(LONG[m], fontsize=8.5, color=INK); ax.legend(fontsize=7, loc="upper left")
    ax.text(0.97, 0.04, lab, transform=ax.transAxes, ha="right", fontweight="bold")
fig.tight_layout(); fig.savefig(FIGS / "Fig06_parity.png"); plt.close(fig)

# Fig. 7: error against dimensionless shut-in time
fig, ax = plt.subplots(figsize=(6.5, 2.8))
te2 = te.copy(); te2["bin"] = pd.cut(te2.dtD, np.logspace(-5, 0.5, 12))
for c, m in zip([INK2, ORANGE, BLUE], ["proxy", "sqrt", "ml_phys"]):
    xm, ym, lo, hi = [], [], [], []
    for bb, g in te2.groupby("bin", observed=True):
        if len(g) < 8:
            continue
        e_ = np.abs((g[m] - g.pbar_srv) / (g.pbar_srv - g.pwf)).dropna()
        xm.append(np.sqrt(bb.left * bb.right)); ym.append(np.median(e_)); lo.append(np.percentile(e_, 25)); hi.append(np.percentile(e_, 75))
    ax.fill_between(xm, lo, hi, color=c, alpha=0.15, lw=0)
    ax.plot(xm, ym, "o-", color=c, lw=1.6, ms=4, label=LONG[m])
ax.axvline(0.03, color=INK2, lw=0.8, ls=(0, (4, 2)))
ax.text(0.034, 0.97, "$\\Delta t_D=0.03$: end of the\nsquare-root regime, Eq. (11)", fontsize=7, color=INK2, va="top")
ax.set_xscale("log"); ax.set_ylim(0, 1.0)
ax.set_xlabel("$\\Delta t_D=\\eta N/L^2$ (cross-validated, binned)"); ax.set_ylabel("|relative error of the deficit|")
ax.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
fig.tight_layout(); fig.savefig(FIGS / "Fig07_identifiability.png"); plt.close(fig)

# Fig. 8: HFTS-1 gauge data
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.9), gridspec_kw=dict(width_ratios=[1.25, 1]))
ax = axes[0]
first = True
for c in HOUR.columns:
    s_ = HOUR[c].dropna().resample("1D").median()
    if "TW" in c:
        ax.plot(s_.index, s_.values, color=ORANGE, lw=1.6, label="slant core well 6TW (between producers)", zorder=3)
    else:
        ax.plot(s_.index, s_.values, color=BLUE, lw=0.7, alpha=0.75, label="11 producers (flowing BHP)" if first else None)
        first = False
ax.axvspan(pd.Timestamp("2016-08-04"), pd.Timestamp("2016-08-14"), color=INK2, alpha=0.12, lw=0)
ax.text(pd.Timestamp("2016-08-20"), 1650, "Aug-2016\ninterference\ntest", ha="left", fontsize=6.5, color=INK2)
ax.set_ylim(0, 4200); ax.set_ylabel("gauge pressure (psia)"); ax.legend(fontsize=7, loc="upper right")
ax.tick_params(axis="x", labelsize=7)
for lab in ax.get_xticklabels():
    lab.set_rotation(30)
ax.text(0.02, 0.04, "(a)", transform=ax.transAxes, fontweight="bold")
ax = axes[1]
for i, (_, b) in enumerate(SUM.sort_values("well").iterrows()):
    cc = CUR[CUR.well == b.well]
    ax.plot(cc.dt_h, cc.pws_psia - b.pwf_last_flowing_psia, color=BLUE, lw=1.0, alpha=0.85, label="ten August-2016 buildups" if i == 0 else None)
gap = SUM.p_core_well_6TW_psia - SUM.pwf_last_flowing_psia
ax.axhspan(gap.min(), gap.max(), color=ORANGE, alpha=0.18, lw=0)
ax.text(0.13, (gap.min() + gap.max()) / 2, "core-well pressure minus $p_{wf}$\n(range over the ten wells)", fontsize=7, color=ORANGE, va="center")
ax.set_xscale("log"); ax.set_xlim(0.1, 120); ax.set_ylim(0, 2400)
ax.set_xlabel("shut-in time $\\Delta t$ (h)"); ax.set_ylabel("$p_{ws}-p_{wf}$ (psi)"); ax.legend(fontsize=7, loc="center left")
ax.text(0.02, 0.96, "(b)", transform=ax.transAxes, fontweight="bold", va="top")
fig.tight_layout(); fig.savefig(FIGS / "Fig08_HFTS1_gauge_data.png"); plt.close(fig)

# Fig. 9: HFTS-1 estimates
F9b = FIELD[FIELD.N > 30].sort_values("well")
fig, ax = plt.subplots(figsize=(6.5, 3.0))
y = np.arange(len(F9b))
ax.barh(y, F9b.p_core_well - F9b.pwf, color=ORANGE, alpha=0.18, height=0.8, label="core-well (6TW) pressure − $p_{wf}$")
ax.plot(F9b.pN - F9b.pwf, y, "s", color=INK2, ms=4.5, label="buildup end (proxy)")
ax.plot(F9b.sqrt_p50 - F9b.pwf, y + 0.18, "^", color=AQUA, ms=4.5, label="square-root decline, Eq. (14)")
ax.errorbar(F9b.ml_phys_p50 - F9b.pwf, y - 0.18, xerr=[F9b.ml_phys_p50 - F9b.ml_phys_lo, F9b.ml_phys_hi - F9b.ml_phys_p50],
            fmt="o", color=BLUE, ms=4.5, capsize=2, lw=1.1, label="ML-P (median, 80 % interval)")
ax.set_yticks(y); ax.set_yticklabels([f"{w} ({n:.0f} h)" for w, n in zip(F9b.well, F9b.N)]); ax.invert_yaxis()
ax.set_xlabel("pressure above the last flowing pressure (psi)")
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.45, -0.24), ncol=2)
fig.tight_layout(); fig.savefig(FIGS / "Fig09_HFTS1_estimates.png"); plt.close(fig)
print("figures written to", FIGS)
print(sorted(p.name for p in FIGS.glob("*.png")))

figures written to /home/claude/hfts1/package_build/HFTS1_Average_Pressure_Package/05_Code/outputs/figures
['Fig01_element_geometry.png', 'Fig02_recovery_law.png', 'Fig03_verification.png', 'Fig04_proxy_bias.png', 'Fig05_estimator_accuracy.png', 'Fig06_parity.png', 'Fig07_identifiability.png', 'Fig08_HFTS1_gauge_data.png', 'Fig09_HFTS1_estimates.png']
